In [10]:
# XGBoost를 이용한 칼로리 예측
# RMSE를 사용하여 평가하기


from __future__ import annotations

import sys
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor


TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
TARGET_COLUMN = "Calories_Burned"


In [2]:
def load_data(train_path: str, test_path: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load train and test datasets."""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    return train_df, test_df

In [12]:
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Preprocess dataset:
    - Drop ID column if exists
    - Encode categorical features
    """
    df = df.copy()

    # Drop the ID column first as it's an identifier and not a feature.
    # The previous error suggested that a column named 'ID' (or similar)
    # with unique values like 'TRAIN_0001' was one-hot encoded.
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])
    elif "id" in df.columns:
        df = df.drop(columns=["id"])

    categorical_cols = df.select_dtypes(include=["object"]).columns
    if len(categorical_cols) > 0:
        df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

    return df

In [4]:
def split_features_target(df: pd.DataFrame, target: str) -> Tuple[pd.DataFrame, pd.Series]:
    """Split features and target."""
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in dataset.")

    X = df.drop(columns=[target])
    y = df[target]
    return X, y



In [5]:
def train_model(X: pd.DataFrame, y: pd.Series) -> XGBRegressor:
    """Train XGBoost Regressor."""
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror",
    )

    model.fit(
        X,
        y,
        verbose=False,
    )

    return model

In [6]:
def evaluate_rmse(model: XGBRegressor, X: pd.DataFrame, y: pd.Series) -> float:
    """Compute RMSE."""
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, preds))
    return rmse

In [13]:
def main() -> None:
    train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)

    train_df = preprocess_data(train_df)
    test_df = preprocess_data(test_df)

    # Diagnosing the missing 'Calories' column
    print(f"Columns in train_df after preprocessing: {train_df.columns.tolist()}")

    X, y = split_features_target(train_df, TARGET_COLUMN)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = train_model(X_train, y_train)

    train_rmse = evaluate_rmse(model, X_train, y_train)
    valid_rmse = evaluate_rmse(model, X_valid, y_valid)

    print(f"Train RMSE: {train_rmse:.4f}")
    print(f"Validation RMSE: {valid_rmse:.4f}")

    if TARGET_COLUMN in test_df.columns:
        X_test, y_test = split_features_target(test_df, TARGET_COLUMN)
        test_rmse = evaluate_rmse(model, X_test, y_test)
        print(f"Test RMSE: {test_rmse:.4f}")
    else:
        print("Test dataset has no target column. Generating predictions only.")
        predictions = model.predict(test_df)
        print(predictions[:10])


if __name__ == "__main__":
    main()

Columns in train_df after preprocessing: ['Exercise_Duration', 'Body_Temperature(F)', 'BPM', 'Height(Feet)', 'Height(Remainder_Inches)', 'Weight(lb)', 'Age', 'Calories_Burned', 'Weight_Status_Obese', 'Weight_Status_Overweight', 'Gender_M']
Train RMSE: 0.3744
Validation RMSE: 1.8338
Test dataset has no target column. Generating predictions only.
[173.83624  193.26514   53.005512 162.13773  225.0232   178.45665
  97.34896   45.90638   78.841354  58.482544]
